# 06 — R3 Cross-product EDA (cointegration / lead-lag / PCA)

Rich-cell version of `notebooks/04_cross_product_eda.py`. Findings doc: `docs/round_3/research/06_cross_product.md`.

Run all cells top-to-bottom. Plots render inline. The final cell is a 30-second TL;DR with every key number from the findings doc.

## Setup

Imports, paths, and constants. `DEAD` strikes have ~0 variance and are excluded from analysis.

In [ ]:
%matplotlib inline
from __future__ import annotations

import glob
import math
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss, coint, grangercausalitytests
from statsmodels.tsa.stattools import acf

warnings.filterwarnings("ignore")

ROOT = Path("/Users/bensinek/Documents/Coding/Prosperity4")
DATA_GLOB = str(ROOT / "data/round_3/prices_round_3_day_*.csv")
PLOTS = ROOT / "docs/round_3/research/plots"
PLOTS.mkdir(parents=True, exist_ok=True)

DEAD = {"VEV_6000", "VEV_6500"}  # variance ~ 0
GOODS = ["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT"]

## Load all-product wide panel

Concatenate the 3 R3 day CSVs (semicolon-separated), pivot to wide form with one column per product, and build a monotonic time index `t`.

In [ ]:
def load_wide() -> pd.DataFrame:
    df = pd.concat([pd.read_csv(f, sep=";") for f in sorted(glob.glob(DATA_GLOB))],
                   ignore_index=True)
    df["t"] = df.day * 1_000_000 + df.timestamp
    wide = (df.pivot_table(index=["day", "timestamp", "t"],
                           columns="product", values="mid_price")
              .reset_index().sort_values("t").reset_index(drop=True))
    return wide


def live_products(wide: pd.DataFrame) -> list[str]:
    cols = [c for c in wide.columns if c not in ("day", "timestamp", "t")]
    return [c for c in cols if c not in DEAD and wide[c].std() > 0]


wide = load_wide()
prods = live_products(wide)
vouchers = [p for p in prods if p.startswith("VEV_")]
rets = np.log(wide[prods]).diff()
print(f"live products ({len(prods)}): {prods}")

## 1. HYDROGEL cointegration vs every other product

Engle-Granger pairwise. Looking for whether HYDROGEL co-moves with anything in levels — informs whether we need a cross-asset hedge.

In [ ]:
def hydrogel_cointegration(wide: pd.DataFrame, prods: list[str]) -> pd.DataFrame:
    rows = []
    h = wide["HYDROGEL_PACK"].dropna()
    for p in prods:
        if p == "HYDROGEL_PACK":
            continue
        s = wide[p].dropna()
        idx = h.index.intersection(s.index)
        try:
            t, pval, _ = coint(h.loc[idx], s.loc[idx])
        except Exception as e:  # noqa: BLE001
            t, pval = np.nan, np.nan
        rows.append({"product": p, "eg_tstat": t, "eg_pvalue": pval})
    out = pd.DataFrame(rows).sort_values("eg_pvalue")
    return out


print("=== 1. Engle-Granger cointegration: HYDROGEL vs each ===")
eg = hydrogel_cointegration(wide, prods)
print(eg.to_string(index=False))

## 6. Stationarity (ADF + KPSS) on levels and returns

Sanity check before reading any cointegration result. ADF H0 = unit root; KPSS H0 = stationary. Both pointing the same direction = clear answer.

In [ ]:
def stationarity_table(wide: pd.DataFrame, prods: list[str]) -> pd.DataFrame:
    rows = []
    for p in prods:
        s = wide[p].dropna()
        r = np.log(s).diff().dropna()
        try:
            adf_lvl = adfuller(s, autolag="AIC")[1]
        except Exception:
            adf_lvl = np.nan
        try:
            kpss_lvl = kpss(s, nlags="auto")[1]
        except Exception:
            kpss_lvl = np.nan
        try:
            adf_ret = adfuller(r, autolag="AIC")[1]
        except Exception:
            adf_ret = np.nan
        try:
            kpss_ret = kpss(r, nlags="auto")[1]
        except Exception:
            kpss_ret = np.nan
        rows.append({"product": p,
                     "adf_lvl_p": adf_lvl, "kpss_lvl_p": kpss_lvl,
                     "adf_ret_p": adf_ret, "kpss_ret_p": kpss_ret})
    return pd.DataFrame(rows)


print("=== 6. Stationarity (ADF p-val for unit root, KPSS p-val for stationarity) ===")
stn = stationarity_table(wide, prods)
print(stn.round(4).to_string(index=False))

## 2. Lead-lag (cross-correlation function)

For each candidate pair, compute CCF over ±50 ticks of returns and pick the peak. A nonzero `peak_lag` would suggest one product leads another.

In [ ]:
def ccf(x: pd.Series, y: pd.Series, max_lag: int = 50) -> tuple[np.ndarray, np.ndarray]:
    """Cross-correlation: corr(x_t, y_{t+lag}). Positive lag => y leads x's future = x leads y."""
    x = (x - x.mean()) / x.std()
    y = (y - y.mean()) / y.std()
    n = len(x)
    lags = np.arange(-max_lag, max_lag + 1)
    out = np.zeros_like(lags, dtype=float)
    for i, k in enumerate(lags):
        if k >= 0:
            out[i] = np.corrcoef(x[: n - k], y[k:])[0, 1] if n - k > 10 else np.nan
        else:
            kk = -k
            out[i] = np.corrcoef(x[kk:], y[: n - kk])[0, 1] if n - kk > 10 else np.nan
    return lags, out


def leadlag_table(rets: pd.DataFrame, pairs: list[tuple[str, str]],
                  max_lag: int = 50) -> pd.DataFrame:
    rows = []
    for a, b in pairs:
        x = rets[a].dropna()
        y = rets[b].dropna()
        idx = x.index.intersection(y.index)
        lags, c = ccf(x.loc[idx], y.loc[idx], max_lag=max_lag)
        valid = ~np.isnan(c)
        if not valid.any():
            continue
        i_peak = np.nanargmax(np.abs(c))
        rows.append({
            "a": a, "b": b,
            "peak_lag": int(lags[i_peak]),
            "peak_corr": float(c[i_peak]),
            "corr_lag0": float(c[lags == 0][0]),
        })
    return pd.DataFrame(rows).sort_values("peak_corr", key=lambda s: s.abs(), ascending=False)


print("=== 2. Lead-lag CCF peaks (returns, \u00b150 ticks) ===")
pairs = []
for v in vouchers:
    pairs.append(("VELVETFRUIT_EXTRACT", v))
for i in range(len(vouchers) - 1):
    pairs.append((vouchers[i], vouchers[i + 1]))
for p in prods:
    if p != "HYDROGEL_PACK":
        pairs.append(("HYDROGEL_PACK", p))
ll = leadlag_table(rets, pairs, max_lag=50)
print(ll.to_string(index=False))

### CCF plot for top 6 pairs by |peak_corr|

Visual confirmation of where the peak sits and how symmetric the CCF is.

In [ ]:
top = ll.head(6)
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, (_, row) in zip(axes.flatten(), top.iterrows()):
    a, b = row["a"], row["b"]
    idx = rets[a].dropna().index.intersection(rets[b].dropna().index)
    lags, c = ccf(rets[a].loc[idx], rets[b].loc[idx], 50)
    ax.bar(lags, c, width=1.0)
    ax.axvline(0, color="k", lw=0.5)
    ax.axhline(0, color="k", lw=0.5)
    ax.set_title(f"{a} vs {b}\npeak_lag={row['peak_lag']} corr={row['peak_corr']:.2f}",
                 fontsize=9)
    ax.set_xlabel("lag (b leads a if lag<0)")
fig.tight_layout()
fig.savefig(PLOTS / "04_leadlag_ccf_top.png", dpi=110)
plt.show()

## 3. Granger causality on top-correlated pairs

Tests whether past values of `a` improve forecasts of `b` over `b`'s own past. Run both directions; bidirectional rejection usually means contemporaneous co-movement leaking into lag structure rather than real causality.

In [ ]:
def granger(rets: pd.DataFrame, a: str, b: str, max_lag: int = 5) -> dict:
    """H0: a does NOT Granger-cause b. Returns p-value at best lag."""
    x = rets[[b, a]].dropna()
    if len(x) < 100:
        return {"a": a, "b": b, "min_p": np.nan, "best_lag": np.nan}
    try:
        res = grangercausalitytests(x, maxlag=max_lag, verbose=False)
        ps = {lag: res[lag][0]["ssr_ftest"][1] for lag in res}
        best = min(ps, key=ps.get)
        return {"a": a, "b": b, "min_p": ps[best], "best_lag": best}
    except Exception:
        return {"a": a, "b": b, "min_p": np.nan, "best_lag": np.nan}


print("=== 3. Granger causality (top pairs, max_lag=5) ===")
g_rows = []
for _, row in ll.head(8).iterrows():
    g_rows.append(granger(rets, row["a"], row["b"], 5))
    g_rows.append(granger(rets, row["b"], row["a"], 5))
gdf = pd.DataFrame(g_rows)
print(gdf.to_string(index=False))

## 4. Voucher pairwise cointegration matrix

Engle-Granger p-values across all live voucher strikes. Heatmap caps at 0.2 so cointegrated pairs (p<0.05) pop visually.

In [ ]:
def voucher_cointegration_matrix(wide: pd.DataFrame,
                                  vouchers: list[str]) -> pd.DataFrame:
    n = len(vouchers)
    M = pd.DataFrame(np.ones((n, n)), index=vouchers, columns=vouchers)
    for i, a in enumerate(vouchers):
        for j, b in enumerate(vouchers):
            if i >= j:
                continue
            x, y = wide[a].dropna(), wide[b].dropna()
            idx = x.index.intersection(y.index)
            try:
                _, p, _ = coint(x.loc[idx], y.loc[idx])
            except Exception:
                p = np.nan
            M.loc[a, b] = p
            M.loc[b, a] = p
    return M


print("=== 4. Voucher cointegration matrix (Engle-Granger p-values) ===")
M = voucher_cointegration_matrix(wide, vouchers)
print(M.round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(M.astype(float), annot=True, fmt=".2f", cmap="viridis_r",
            vmin=0, vmax=0.2, ax=ax, cbar_kws={"label": "EG p-value (capped 0.2)"})
ax.set_title("Voucher pairwise Engle-Granger cointegration p-values\n(<0.05 = cointegrated)")
fig.tight_layout()
fig.savefig(PLOTS / "04_voucher_coint_heatmap.png", dpi=110)
plt.show()

cointegrated_pairs = []
for i, a in enumerate(vouchers):
    for j, b in enumerate(vouchers):
        if i < j and M.loc[a, b] < 0.05:
            cointegrated_pairs.append((a, b, M.loc[a, b]))
print("Cointegrated voucher pairs (p<0.05):", cointegrated_pairs)

## 5. PCA on returns

Eigendecompose the standardized return covariance. PC1 should capture the common VFE/voucher level factor; subsequent PCs reveal whether HYDROGEL lives on its own axis and whether wing/curvature factors exist.

In [ ]:
def pca_returns(rets: pd.DataFrame, prods: list[str]) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    R = rets[prods].dropna()
    Z = (R - R.mean()) / R.std()
    cov = np.cov(Z.values.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    explained = eigvals / eigvals.sum()
    loadings = pd.DataFrame(eigvecs, index=prods,
                            columns=[f"PC{i+1}" for i in range(len(prods))])
    return explained, eigvals, loadings


print("=== 5. PCA on returns ===")
explained, eigvals, loadings = pca_returns(rets, prods)
print("Explained variance ratio:", np.round(explained, 4))
print("Cumulative:", np.round(np.cumsum(explained), 4))
n95 = int(np.argmax(np.cumsum(explained) >= 0.95) + 1)
print(f"# factors for >=95% variance: {n95}")
print("\nLoadings PC1..PC4:")
print(loadings.iloc[:, :4].round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(range(1, len(explained) + 1), explained)
axes[0].plot(range(1, len(explained) + 1), np.cumsum(explained), "o-r")
axes[0].axhline(0.95, color="grey", ls="--")
axes[0].set_xlabel("PC")
axes[0].set_ylabel("explained var ratio")
axes[0].set_title("Scree (bars) + cumulative (red)")
sns.heatmap(loadings.iloc[:, :4], annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, ax=axes[1])
axes[1].set_title("PC1..PC4 loadings")
fig.tight_layout()
fig.savefig(PLOTS / "04_pca_scree_loadings.png", dpi=110)
plt.show()

## 7. Delta-weighted voucher basket vs VFE

Build basket = Σ(δᵢ · voucher_midᵢ) using BS deltas at σ=0.15, then look at spread vs VFE. Stationary spread would suggest a synthetic-underlying trade.

In [ ]:
def bs_call_delta(S: float, K: float, T: float, sigma: float) -> float:
    if T <= 0 or sigma <= 0 or S <= 0:
        return 1.0 if S > K else 0.0
    d1 = (math.log(S / K) + 0.5 * sigma * sigma * T) / (sigma * math.sqrt(T))
    return 0.5 * (1 + math.erf(d1 / math.sqrt(2)))


def basket_spread(wide: pd.DataFrame) -> pd.DataFrame:
    """Build basket = sum(delta_i * voucher_mid_i) and compare to VFE."""
    sigma = 0.15
    # TTE per day in years (R1=7d expiry, days 0/1/2 -> tte 8,7,6 days)
    tte_per_day = {0: 8 / 365, 1: 7 / 365, 2: 6 / 365}
    strikes = [4000, 4500, 5000, 5100, 5200, 5300, 5400, 5500]
    vouchers = [f"VEV_{k}" for k in strikes]
    sub = wide[["day", "t", "VELVETFRUIT_EXTRACT"] + vouchers].dropna().copy()
    deltas = pd.DataFrame(index=sub.index, columns=vouchers, dtype=float)
    for i, row in sub.iterrows():
        T = tte_per_day[int(row["day"])]
        S = row["VELVETFRUIT_EXTRACT"]
        for K, v in zip(strikes, vouchers):
            deltas.at[i, v] = bs_call_delta(S, K, T, sigma)
    basket = (deltas.values * sub[vouchers].values).sum(axis=1)
    sub["basket"] = basket
    sub["spread"] = sub["basket"] - sub["VELVETFRUIT_EXTRACT"]
    return sub[["day", "t", "VELVETFRUIT_EXTRACT", "basket", "spread"]]


print("=== 7. Delta-weighted voucher basket vs VFE (sigma=0.15) ===")
bsk = basket_spread(wide)
sp = bsk["spread"]
try:
    adf_p = adfuller(sp, autolag="AIC")[1]
except Exception:
    adf_p = np.nan
print(f"basket mean={bsk['basket'].mean():.2f}, VFE mean={bsk['VELVETFRUIT_EXTRACT'].mean():.2f}")
print(f"spread mean={sp.mean():.2f}, std={sp.std():.2f}, ADF p={adf_p:.4f}")

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(bsk["t"], bsk["basket"], label="basket(\u03a3\u03b4\u00b7voucher)", lw=0.6)
axes[0].plot(bsk["t"], bsk["VELVETFRUIT_EXTRACT"], label="VFE", lw=0.6)
axes[0].legend(); axes[0].set_title("Basket vs VFE")
axes[1].plot(bsk["t"], sp, lw=0.4)
axes[1].axhline(sp.mean(), color="r", ls="--")
axes[1].set_title(f"Spread = basket \u2212 VFE  (ADF p={adf_p:.4f})")
fig.tight_layout()
fig.savefig(PLOTS / "04_basket_vs_vfe.png", dpi=110)
plt.show()

## 8. Volatility clustering (ACF of squared returns)

Lag-1/5/10 autocorrelation of r² flags GARCH-type behavior. High values in deep-ITM strikes are expected (gamma toggles with VFE level).

In [ ]:
def vol_clustering(rets: pd.DataFrame, prods: list[str], nlags: int = 20) -> pd.DataFrame:
    rows = []
    for p in prods:
        r = rets[p].dropna()
        if len(r) < 100:
            continue
        # Ljung-Box on r^2
        sq = r ** 2
        ac = acf(sq, nlags=nlags, fft=True)
        rows.append({"product": p, "acf_sq_lag1": ac[1],
                     "acf_sq_lag5": ac[5], "acf_sq_lag10": ac[10]})
    return pd.DataFrame(rows)


print("=== 8. Volatility clustering (ACF of squared returns) ===")
vc = vol_clustering(rets, prods)
print(vc.round(3).to_string(index=False))

## 9. Day-to-day stability

Per-day return correlation matrices on a focus set. If correlations are stable across days 0/1/2, signals generalize; if not, beware regime shifts.

In [ ]:
def per_day_corr(wide: pd.DataFrame, prods: list[str]) -> dict:
    out = {}
    for d in sorted(wide["day"].unique()):
        sub = wide[wide["day"] == d]
        rets = np.log(sub[prods]).diff()
        out[int(d)] = rets.corr()
    return out


print("=== 9. Per-day return correlations: VFE \u00d7 VEV_5000 / VEV_5200 / HYDROGEL ===")
per_day = per_day_corr(wide, prods)
focus = ["VELVETFRUIT_EXTRACT", "VEV_5000", "VEV_5200", "HYDROGEL_PACK"]
for d, C_ in per_day.items():
    present = [p for p in focus if p in C_.columns]
    print(f"--- day {d} ---")
    print(C_.loc[present, present].round(2).to_string())

## Quick numerical recap

Min p-values, factor counts, spread ADF, top lead-lag — pulled from the objects computed above for a one-glance summary before reading the TL;DR.

In [ ]:
print("### FINDINGS")
print("- HYDROGEL_PACK Engle-Granger min p-value across all other live products:",
      f"{eg['eg_pvalue'].min():.4f}")
print(f"- Cointegrated voucher pairs (EG p<0.05): {len(cointegrated_pairs)}")
print(f"- # PCs for 95% variance: {n95}")
print(f"- Basket-VFE spread ADF p: {adf_p:.4f}")
print("- Top lead-lag (peak_lag != 0):",
      ll[ll['peak_lag'] != 0].head(3).to_dict("records"))

## Summary + Key Findings

30-second TL;DR — every number and observation from `docs/round_3/research/06_cross_product.md`.

### Top-line
- **HYDROGEL is independent.** Treat as standalone MM book. No cross-asset hedge needed.
- **No exploitable lead-lag.** Every meaningful pair peaks at lag 0; off-zero CCF magnitudes ≤ 0.02 (noise level).
- **No clean cross-product stat-arb.** Voucher↔voucher cointegration is dominated by deep-ITM pairs (mechanical) and dead-strike VEV_5500 (low-variance artifact).
- **Factor structure**: PC1 = 60% = single VFE/voucher factor; **PC2 = 10% = HYDROGEL alone** — its own orthogonal axis. Two separable trading books.
- **All structure is stable across days 0/1/2** to 2 decimals. Low generalisation risk for any signal found.

### 1. HYDROGEL independence (confirmed)
- Engle-Granger cointegration with each other product: **p < 0.0001 for all**.
- ADF on HYDROGEL levels: **p = 0.0000** → stationary.
- KPSS on HYDROGEL levels: **p = 0.01** → reject stationarity.
- 1-tick return correlation with anything: **~0.00**.
- Max |CCF| over ±50 lags vs every other product: **≤ 0.020** (noise).
- Per-day return corr (days 0/1/2): identical, all ~0.00.
- EG p<0.0001 looks alarming but is **degenerate** — HYDROGEL already mean-reverts in levels (ADF rejects unit root), so EG residual is trivially stationary. Combined with zero return correlation and zero CCF, no real economic linkage.
- HYDROGEL = pure independent MM target; PC2 confirms it occupies its own dimension.

### 2. Lead-lag (nothing actionable)
- VFE↔voucher and voucher↔voucher: peak |corr| always at **lag 0** (range **0.66–0.91**). CCF symmetric and decaying — pure contemporaneous co-movement.
- HYDROGEL vs everything: peak |corr| over ±50 lags ranges **0.01–0.02**. Indistinguishable from noise.
- Granger tests on top pairs (max lag 5) reject H0 in **both directions** with p≈0 — picking up persistence in contemporaneous returns, not directional info.
- No "this voucher leads VFE by N ticks" signal. Quoting one product off another's stale mid will not produce alpha.

### 3. Voucher cointegration (limited usefulness)
- **VEV_4000 ↔ VEV_4500: p ≈ 0.000** — deep-ITM, both ~delta-1 on VFE. Cointegrated by construction (≈ VFE − K + small extrinsic). Not free alpha.
- **VEV_5500 ↔ everything: p < 0.05.** VEV_5500 is near-dead (low variance, like VEV_6000/6500). Cointegration is a low-variance artifact — treat with suspicion.
- **Belly strikes 4500–5400**: pairwise **p ≈ 0.10–0.89** — not cointegrated in levels. Idiosyncratic per strike; voucher-chain agent should model via IV smile.
- Recommend deferring voucher↔voucher RV trades to the voucher chain agent (BS / IV machinery). Only obvious raw-price candidate is **VEV_4000 / VEV_4500 spread = strike-difference parity** (trade if it deviates from the constant 500).

### 4. Basket vs underlying
- Delta-weighted basket Σ(δᵢ·Vᵢ) at σ=0.15: **mean ≈ 2489** vs VFE mean **≈ 5250**.
- Spread = basket − VFE: **mean − 2761**, **std 62**, **ADF p = 0.000** (stationary).
- Mean is intrinsic value of the deep strikes; construction is not synthetic-underlying replication. Stationarity is mean reversion of a quantity with no economic equivalence to VFE.
- Not directly tradeable. Proper synthetic underlying via put-call parity / strike spread is the voucher-chain agent's job.

### 5. PCA on returns (10 live products)
Explained variance: **PC1=60.5%, PC2=10.0%, PC3=8.5%, PC4=6.4%, …, 7 PCs for ≥95%**.

| PC | Loading pattern | Interpretation |
|---|---|---|
| PC1 (60%) | uniform negative on VFE + all vouchers, ≈0 on HYDROGEL | **VFE/voucher level factor** |
| PC2 (10%) | **−1.00 on HYDROGEL**, ≈0 on everything | **HYDROGEL standalone factor** |
| PC3 (8.5%) | −0.95 on VEV_5500, small on others | wing/skew (mostly the dead-ish 5500) |
| PC4 (6.4%) | −0.80 on VEV_5400, +0.32 / +0.38 on 4000 / 4500 | curvature (deep-ITM vs OTM-wing) |
| PC5–PC10 | per-strike idiosyncratic | residual noise |

- Board cleanly decomposes into **{VFE+vouchers} ⊥ {HYDROGEL}**. No hidden joint factor.
- Long tail (7 PCs to 95%) means each voucher carries non-trivial idiosyncratic noise — no broad voucher-residual mean reversion to harvest.

### 6. Stationarity (sanity check)
- ADF p ≈ 0 on **levels** for every product (consistent with finite ranges + tick-scale mean reversion / negative lag-1 autocorr).
- KPSS rejects stationarity on the same level series → mean-reverting but with slow regime drift.
- Returns: ADF p=0, KPSS p=0.10 → stationary as expected.

### 7. Vol clustering (ACF of r², lag 1)
- **VEV_4000 = 0.41**, **VEV_4500 = 0.35**, **VEV_5500 = 0.31**, **VEV_5400 = 0.16**, **HYDROGEL = 0.13**, **VFE = 0.11**.
- Strong clustering only in deep-ITM and far-wing strikes — inherited from VFE level moves under option Greeks (gamma flips on/off as strike crosses spot).
- HYDROGEL/VFE clustering is mild — no need for GARCH-style adaptive sizing in a first iteration.

### 8. Day-to-day stability
- Per-day return correlation matrices for {VFE, VEV_5000, VEV_5200, HYDROGEL} are **identical to 2 decimals across days 0/1/2**.
- Cross-product structure is stable. (Whether absolute IVs / smile shape are stable is a separate question for the voucher agent.)

### Recommended cross-product strategies
1. **HYDROGEL: pure standalone MM.** No need to look at any other book. Confirmed by both correlation and PCA.
2. **VFE: standalone delta-1 product**, but the voucher chain agent's hedging flow will hit this book — coordinate inventory limits, not strategy.
3. **No cross-product alpha overlay**: no lead-lag signal to forecast voucher from VFE (or vice versa) at 1-tick scale. Don't waste a slot on it.
4. **VEV_4000 / VEV_4500 strike-spread guardrail**: if the spread between these adjacent deep-ITM vouchers ever deviates meaningfully from 500 (the strike difference), that's a put-call-parity-style arb. Worth a sanity check, low-EV.
5. **Defer voucher RV to the voucher-chain agent.** No clean voucher↔voucher cointegration in the belly (5000–5400) — IV-space modelling required.